# Lab 11: From Kafka to PostgreSQL

Run the cells one at a time. A generator publishes random order events to Kafka without stopping, and two consumers insert them into PostgreSQL. You watch the tables grow, stop the consumers, restart them, and see which events end up twice and how to prevent it.

The generator and the consumers are the scripts of this folder, and the notebook starts them as separate processes, so that stopping one is the same as a crash.

## Connect to PostgreSQL

PostgreSQL is the local service of the development container, and Kafka is the one of lab 10. The logs of the scripts go to `labs/shared/output/lab11`.

In [ ]:
import subprocess
import time
from pathlib import Path

import pandas as pd
import psycopg

pg = psycopg.connect(autocommit=True)
LOGS = Path('../shared/output/lab11')
LOGS.mkdir(parents=True, exist_ok=True)
print(pg.info.parameter_status('server_version'))

## The helpers

`query` runs a SQL statement and returns the rows as a table. `start` runs a script as a process of its own, with its output in a log, and `stop` kills processes, like a crash would. `describe` prints the partitions of the topic and the position of a group, with the `describe.py` of lab 10. `counts` and `watch` show how many rows each table has.

In [ ]:
def query(statement):
    with pg.cursor() as cur:
        cur.execute(statement)
        columns = [column.name for column in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)


def start(name, script, *args):
    log = open(LOGS / f'{name}.log', 'w', encoding='utf-8')
    return subprocess.Popen(['python3', '-u', script, *args],
                            stdout=log,
                            stderr=subprocess.STDOUT)


def stop(*processes):
    for process in processes:
        process.kill()
        process.wait()


def describe(group):
    result = subprocess.run(
        ['python3', '../10-kafka/describe.py', '--group', group],
        capture_output=True,
        text=True,
        check=True)
    print(result.stdout)


def counts():
    return query('''
        SELECT 'order_events' AS table_name,
               count(*) AS rows,
               count(DISTINCT event_id) AS distinct_events
        FROM lab_11.order_events
        UNION ALL
        SELECT 'order_events_unique', count(*), count(DISTINCT event_id)
        FROM lab_11.order_events_unique
    ''')


def watch(seconds, every=3):
    for _ in range(seconds // every):
        time.sleep(every)
        rows = counts()['rows'].tolist()
        print(f'{time.strftime("%H:%M:%S")}  order_events: {rows[0]}  '
              f'order_events_unique: {rows[1]}')

## Create the tables and the topic

Both tables have the same columns: the event, and the partition and the offset of Kafka where it was read. `order_events` has no key, so it accepts the same event twice. In `order_events_unique`, `event_id` is the primary key, so it does not: its consumer ignores an event that is already there, and a consumer that gives the same result when it processes an event twice as when it processes it once is idempotent. The topic `orders` is created again, empty, with 3 partitions.

In [ ]:
pg.execute('DROP SCHEMA IF EXISTS lab_11 CASCADE')
pg.execute('CREATE SCHEMA lab_11')
for table, key in [('order_events', ''),
                   ('order_events_unique', 'PRIMARY KEY')]:
    pg.execute(f'''
        CREATE TABLE lab_11.{table} (
            event_id uuid {key},
            event_type text,
            order_id bigint,
            restaurant text,
            amount numeric(6, 2),
            occurred_at timestamptz,
            kafka_partition integer,
            kafka_offset bigint)
    ''')

subprocess.run(['python3', '../10-kafka/topic.py', 'delete'],
               capture_output=True)
time.sleep(3)
result = subprocess.run(['python3', '../10-kafka/topic.py', 'create'],
                        capture_output=True,
                        text=True,
                        check=True)
print(result.stdout)

## Start the generator and two consumers

The generator publishes about two new orders per second, and each order goes through `order_created`, `order_confirmed` and `order_delivered`, or `order_cancelled`. The two consumers are in different groups, so each one reads all the events. `sink-plain` writes to `order_events`. `sink-unique` writes to `order_events_unique` and ignores an event that is already there. Both confirm their position in Kafka every 50 events.

The cell prints the number of rows of each table every 3 seconds.

In [ ]:
generator = start('generator', 'generator.py')
plain = start('sink-plain', 'sink.py', '--group', 'sink-plain', '--name',
              'plain', '--table', 'order_events')
unique = start('sink-unique', 'sink.py', '--group', 'sink-unique', '--name',
               'unique-1', '--table', 'order_events_unique', '--idempotent')
watch(15)

## Look at the rows

The last events that arrived in `order_events`, with the partition and the offset where each one was read.

In [ ]:
query('''
    SELECT order_id, event_type, restaurant, amount, kafka_partition, kafka_offset
    FROM lab_11.order_events
    ORDER BY occurred_at DESC
    LIMIT 8
''')

## Turn the events into the state of each order

Each event says what happened to an order. The last event of each order says where it is now. The query uses `DISTINCT ON` to keep only that one, and shows the 10 newest orders.

In [ ]:
query('''
    SELECT *
    FROM (SELECT DISTINCT ON (order_id) order_id, restaurant, amount,
                 event_type AS status
          FROM lab_11.order_events_unique
          ORDER BY order_id, occurred_at DESC) AS last_event
    ORDER BY order_id DESC
    LIMIT 10
''')

## Stop the consumers

`stop` kills both consumers, as if their computers had failed. The generator keeps publishing. For 15 seconds the tables do not grow, and the lag of the group, the events that Kafka holds and the group has not read, does.

In [ ]:
stop(plain, unique)
watch(15)
describe('sink-unique')

## Start the consumers again

The consumers continue from the last position they confirmed. The tables catch up with the events published while they were stopped. The consumers had read some events after their last confirmation and, as they were killed, they did not confirm them, so they read them again. The table `order_events` inserts them again, and `order_events_unique` ignores them, because its consumer is idempotent.

In [ ]:
plain = start('sink-plain', 'sink.py', '--group', 'sink-plain', '--name',
              'plain', '--table', 'order_events')
unique = start('sink-unique', 'sink.py', '--group', 'sink-unique', '--name',
               'unique-1', '--table', 'order_events_unique', '--idempotent')
watch(21)
counts()

## The events that arrived twice

An event that was read twice has the same partition and the same offset in two rows. The table `order_events_unique` has none.

In [ ]:
query('''
    SELECT kafka_partition, kafka_offset, count(*) AS times
    FROM lab_11.order_events
    GROUP BY kafka_partition, kafka_offset
    HAVING count(*) > 1
    ORDER BY kafka_partition, kafka_offset
    LIMIT 10
''')

## Add a second consumer to a group

A second consumer joins the group `sink-unique`. Kafka shares the 3 partitions between the two, and the table keeps growing without repeated events. `describe` shows the members of the group and their partitions.

In [ ]:
unique_2 = start('sink-unique-2', 'sink.py', '--group', 'sink-unique', '--name',
                 'unique-2', '--table', 'order_events_unique', '--idempotent')
time.sleep(15)
describe('sink-unique')
watch(9)
counts()

## Reprocess every event

Stop the generator and the consumers, and wait 12 seconds, so that Kafka notices that the consumers are gone. `reset_offsets.py` moves the confirmed position of each group to the first event of the topic, as when an error in the processing is fixed and the events have to be processed again. Then each consumer runs until it has no events left. `order_events` has all the events a second time, and `order_events_unique` does not change.

In [ ]:
stop(generator, plain, unique, unique_2)
time.sleep(12)
before = counts()
for group, name, table, *flags in [('sink-plain', 'plain', 'order_events'),
                                   ('sink-unique', 'unique-1',
                                    'order_events_unique', '--idempotent')]:
    subprocess.run(['python3', 'reset_offsets.py', '--group', group],
                   check=True)
    subprocess.run([
        'python3', 'sink.py', '--group', group, '--name', name, '--table',
        table, '--stop-when-idle', '8', *flags
    ],
                   check=True)
pd.concat([before, counts()], axis=1, keys=['before', 'after'])

## Clean up

The last cell drops the schema, deletes the topic, closes the connection and deletes the logs.

In [ ]:
pg.execute('DROP SCHEMA lab_11 CASCADE')
pg.close()
subprocess.run(['python3', '../10-kafka/topic.py', 'delete'], check=True)
for log in LOGS.glob('*.log'):
    log.unlink()